# AfriVoices — GÉNÉRATEUR FINAL : soumission par (langue × type)

**C'est le notebook qui produit la soumission finale (leaderboard 0.39477).**
Config figée : `baobab-asr-v9-2` + KenLM v2 + **(α, β) = (0.5, 0.0)** sur les six langues
et les deux types. Historique : le même générateur à (0.7, 0.5) = soumission de contrôle
**0.39923** ; le passage à (0.5, 0.0) est justifié dans le RAPPORT (Partie II, §4 —
test à dominante spontanée, deux grilles concordantes).

Propriétés du moteur (vs le générateur historique) :
1. **Décodage direct pleine longueur** (le fenêtrage des clips > 20s a été mesuré
   perdant — voir Partie II §3) ; secours OOM : fp16 puis fenêtres+recollage de logits.
2. **Troncature anti-padding** des logits (frames de padding jamais entraînées).
3. **Routage par (langue × type)** : le test fournit la colonne `type` par clip ;
   les deux types partagent ici les mêmes (α, β), le routage reste câblé.
4. **Décodage LM parallèle** (`decode_batch`, sorties identiques au séquentiel).

**Mode d'emploi** : §1 (install + REDÉMARRER) → §2 → §3 → §4 → §5 (reprise auto tous
les 10 parquets) → §6 (asserts + CSV, 41 733 lignes).
Conformité edge : même modèle, même LM ; RTF par clip inchangé (le parallélisme
n'accélère que la production du CSV).

## 1. Dépendances (redémarrer le runtime après la 1ère exécution)

In [ ]:
import importlib, subprocess, sys
need = [m for m in ["kenlm", "pyctcdecode"] if importlib.util.find_spec(m) is None]
if need:
    print("installation de", need, "...")
    # IMPORTANT : pyctcdecode épingle numpy<2 ; l'installer avec ses dépendances RÉTROGRADE
    # le numpy de Colab et laisse une installation mixte 1.x/2.x -> au prochain import :
    # "numpy.dtype size changed, may indicate binary incompatibility".
    # On installe donc SANS dépendances (pygtrie = sa seule vraie dépendance).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyctcdecode", "pygtrie"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime (Exécution > Redémarrer la session), puis relance cette cellule.")
else:
    import numpy
    print(f"✓ kenlm + pyctcdecode disponibles (numpy {numpy.__version__}, intact) — continue en §2")

## 2. CONFIG — le seul bloc à modifier

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v9-2"
LM_SUBDIR  = "lm_v2"
TAG = "v9_2_polB"

# (alpha, beta) par TYPE puis par LANGUE — CONFIG FINALE (leaderboard 0.39477).
# Pour reproduire la soumission de contrôle (0.39923) : tout remettre à (0.7, 0.5).
AB = {
  "scripted":   {"swa": (0.5, 0.0), "kik": (0.5, 0.0), "luo": (0.5, 0.0),
                 "kln": (0.5, 0.0), "mas": (0.5, 0.0), "som": (0.5, 0.0)},
  "unscripted": {"swa": (0.5, 0.0), "kik": (0.5, 0.0), "luo": (0.5, 0.0),
                 "kln": (0.5, 0.0), "mas": (0.5, 0.0), "som": (0.5, 0.0)},
}
TYPE_DEFAUT = "scripted"            # clip sans type reconnu -> ce type
UNIGRAM_TOP = 50000
SEUIL_DIRECT_S = None   # None = décodage direct partout (config de la soumission, GPU/grande RAM).
                        # Edge CPU ≤ 8 Go : mettre 60 -> les clips plus longs passent d'office par le
                        # repli fenêtré (seuil mesuré : voir validation/HARDWARE_VALIDATION_*.md).

# fenêtrage — UNIQUEMENT en secours OOM (pas de seuil de durée : tout passe en direct)
WIN_SEC, OVER_SEC, SEARCH_SEC = 15.0, 2.0, 3.0
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
LM    = f"{BASE}/{LM_SUBDIR}"
PARTIEL = f"{BASE}/submission_{TAG}_partiel.csv"
FINAL   = f"{BASE}/submission_{TAG}.csv"
assert os.path.isdir(MODEL), f"modèle absent: {MODEL}"
assert os.path.isdir(LM), f"LM absent: {LM}"
print(f"modèle: {MODEL_NAME} | LM: {LM_SUBDIR}")
for typ, d in AB.items():
    print(f"  {typ}: " + " ".join(f"{l}={ab}" for l, ab in d.items()))
print(f"sortie: {FINAL}")

## 3. Rechargement : modèle + labels + décodeurs par (langue × type) + decode_robuste

In [ ]:
import torch, io, base64, contextlib, numpy as np, soundfile as sf, librosa, tempfile
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "" if device == "cuda" else "⚠️ GPU recommandé (30-45 min vs plusieurs heures)")

model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)

raw = [t for t, _ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n = [], 0
for t in raw:
    if t == blank_tok: labels.append("")
    elif t in ("|", " "): labels.append(" ")
    elif t in {"[UNK]", "<s>", "</s>"}: n += 1; labels.append("\u2047" * n)
    else: labels.append(t)
assert labels.count("") == 1 and labels.count(" ") == 1

def decode_robuste(b):
    if not b: raise ValueError("vide")
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception:
        try: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
        except Exception:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                try: tf.write(base64.b64decode(b))
                except Exception: tf.write(b)
                p = tf.name
            arr, sr = librosa.load(p, sr=16000, mono=True); os.unlink(p)
            return arr.astype(np.float32)
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def unigrams(lang, top):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w, _ in c.most_common(top)]

UNI = {l: unigrams(l, UNIGRAM_TOP) for l in ["swa", "kik", "luo", "kln", "mas", "som"]}

# un décodeur par config distincte (langue, alpha, beta) — dédupliqué
_deco_cache = {}
def _deco(lang, a, b):
    key = (lang, a, b)
    if key not in _deco_cache:
        _deco_cache[key] = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                                            unigrams=UNI[lang], alpha=a, beta=b)
    return _deco_cache[key]

decoders = {}   # (type, lang) -> décodeur
for typ, d in AB.items():
    for lang, (a, b) in d.items():
        decoders[(typ, lang)] = _deco(lang, a, b)
print(f"✓ prêt : {MODEL_NAME} + {LM_SUBDIR} — {len(_deco_cache)} décodeurs distincts pour {len(decoders)} routes")
print("(warnings pyctcdecode 'length > 1' / 'unigrams don't agree' = bénins)")

## 4. Données de test (Drive si présent, sinon téléchargement HF)

In [ ]:
import glob
TEST_CANDIDATES = [f"{BASE}/test", "/content/test_data"]
parquets = []
for td in TEST_CANDIDATES:
    parquets = sorted(glob.glob(f"{td}/**/*.parquet", recursive=True))
    if parquets:
        TEST_DIR = td; break
if not parquets:
    print("test absent localement -> téléchargement HF...")
    from huggingface_hub import snapshot_download
    TEST_DIR = "/content/test_data"
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir=TEST_DIR)
    parquets = sorted(glob.glob(f"{TEST_DIR}/**/*.parquet", recursive=True))
print(f"{len(parquets)} parquets test dans {TEST_DIR}")
assert parquets, "aucun parquet test trouvé"

## 5. Inférence : direct pleine longueur, routage (langue × type), secours OOM → fp16 → fenêtres

In [ ]:
import pandas as pd, time
from tqdm.auto import tqdm

MAX_SEC_BATCH = 160.0

def _forward(feats, fp16=False):
    ctx = torch.autocast("cuda", dtype=torch.float16) if (fp16 and device == "cuda") else contextlib.nullcontext()
    with torch.no_grad(), ctx:
        return model(**{k: v.to(device) for k, v in feats.items()}).logits.float().cpu().numpy()

# ---------- fenêtrage v2 (secours OOM uniquement) ----------
def _rms(x, fl=400, hop=160):
    if len(x) < fl:
        return np.array([float(np.sqrt(np.mean(x ** 2) + 1e-12))]), hop
    nfr = 1 + (len(x) - fl) // hop
    idx = np.arange(nfr)[:, None] * hop + np.arange(fl)[None, :]
    fr = x[idx]
    return np.sqrt((fr ** 2).mean(axis=1) + 1e-12), hop

def fenetres_v2(arr, sr=16000):
    L = len(arr); w = int(WIN_SEC * sr); o = int(OVER_SEC * sr); s_s = int(SEARCH_SEC * sr)
    if L <= w: return [(0, L)]
    wins = []; cur = 0
    while L - cur > w:
        target = cur + w
        a = max(target - s_s, cur + o)
        seg = arr[a:min(target, L)]
        rms, hop = _rms(seg)
        cut = a + int(np.argmin(rms)) * hop + 200
        cut = min(max(cut, cur + o), L - 1)
        wins.append((cur, min(cut + o // 2, L)))
        cur = max(cut - o // 2, 0)
    wins.append((max(0, L - w), L))
    if len(wins) >= 2 and wins[-1][0] <= wins[-2][0]:
        wins[-2] = (wins[-2][0], L); wins.pop()
    return wins

def stitch(wins, lgs):
    """Tranchage au milieu de chaque chevauchement (posé sur la coupe-silence) :
    chaque frame provient d'une seule fenêtre, indices relatifs à la fenêtre -> zéro dérive."""
    if len(wins) == 1:
        return lgs[0].astype(np.float32)
    parts = []
    for i, ((s, e), lg) in enumerate(zip(wins, lgs)):
        nfr = lg.shape[0]
        if nfr == 0: continue
        spf = (e - s) / nfr
        lo = s if i == 0 else (wins[i - 1][1] + s) // 2
        hi = e if i == len(wins) - 1 else (e + wins[i + 1][0]) // 2
        a = max(0, int(round((lo - s) / spf)))
        b = min(nfr, int(round((hi - s) / spf)))
        if b > a:
            parts.append(lg[a:b].astype(np.float32))
    return np.concatenate(parts, axis=0)

def _solo(arr):
    """Un clip : direct fp32 -> direct fp16 -> fenêtres v2 + stitch. Retourne les logits tronqués."""
    if SEUIL_DIRECT_S and len(arr) / 16000.0 > SEUIL_DIRECT_S:
        return _par_fenetres(arr)
    fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt")
    try:
        return _forward(fe)[0]
    except RuntimeError:
        torch.cuda.empty_cache()
    try:
        return _forward(fe, fp16=True)[0]
    except RuntimeError:
        torch.cuda.empty_cache()
    return _par_fenetres(arr)

def _par_fenetres(arr):
    wins = fenetres_v2(arr)
    lgs = []
    for (ws, we) in wins:
        f1 = processor.feature_extractor([arr[ws:we]], sampling_rate=16000, return_tensors="pt")
        lgs.append(_forward(f1, fp16=True)[0])
    return stitch(wins, lgs)

def logits_batch(arrs):
    """Logits TRONQUÉS (anti-padding) ; si un batch OOM, bascule item par item via _solo."""
    order = sorted(range(len(arrs)), key=lambda k: len(arrs[k]))
    out = [None] * len(arrs)
    if SEUIL_DIRECT_S:
        for k in order:
            if len(arrs[k]) / 16000.0 > SEUIL_DIRECT_S:
                out[k] = _par_fenetres(arrs[k])
        order = [k for k in order if out[k] is None]
    i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and (j - i + 1) * (len(arrs[order[j]]) / 16000.0) <= MAX_SEC_BATCH:
            j += 1
        idxs = order[i:j]
        feats = processor.feature_extractor([arrs[k] for k in idxs], sampling_rate=16000,
                                            return_tensors="pt", padding=True)
        try:
            lg = _forward(feats)
            am = feats.get("attention_mask")
            T_out = lg.shape[1]
            if am is not None:
                fr = am.sum(-1).numpy().astype(float); fmax = float(am.shape[1])
            else:
                fr = np.array([len(arrs[k]) for k in idxs], dtype=float); fmax = fr.max()
            for bi, k in enumerate(idxs):
                n_true = max(1, int(round(T_out * fr[bi] / fmax)))
                out[k] = lg[bi, :min(n_true, T_out)]
        except RuntimeError:
            torch.cuda.empty_cache()
            for k in idxs:
                out[k] = _solo(arrs[k])
        i = j
    return out

# ---------- boucle principale (reprise auto) ----------
done = {}
if os.path.exists(PARTIEL):
    dfp = pd.read_csv(PARTIEL)
    done = dict(zip(dfp["id"].astype(str), zip(dfp["language"], dfp["transcription"])))
    print("reprise :", len(done), "déjà transcrits")

rows = [{"id": k, "language": v[0], "transcription": v[1]} for k, v in done.items()]
t0 = time.time()

for pi, pq in enumerate(tqdm(parquets, desc="Inférence", unit="pq")):
    df = pd.read_parquet(pq)
    buckets = {}   # (lang, type) -> [(rid, arr)]
    for _, r in df.iterrows():
        rid = str(r["id"])
        if rid in done: continue
        lang = r.get("language")
        typ = str(r.get("type") or "").lower().strip()
        if typ not in AB: typ = TYPE_DEFAUT
        b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
        try:
            arr = decode_robuste(b)
        except Exception:
            rows.append({"id": rid, "language": lang, "transcription": "_"})
            continue
        buckets.setdefault((lang, typ), []).append((rid, arr))

    for (lang, typ), its in buckets.items():
        dec = decoders.get((typ, lang)) or next(iter(decoders.values()))
        lgs = logits_batch([arr for _, arr in its])
        for (rid, _), lg in zip(its, lgs):
            try:
                txt = dec.decode(lg)
            except Exception:
                txt = "_"
            rows.append({"id": rid, "language": lang, "transcription": txt})

    if (pi + 1) % 10 == 0 or pi == len(parquets) - 1:
        pd.DataFrame(rows).to_csv(PARTIEL, index=False)
        print(f"parquet {pi + 1}/{len(parquets)} | {len(rows)} transcrits | {(time.time() - t0) / 60:.0f} min")

print("✓ inférence par type terminée :", len(rows))

## 6. Assemblage final + asserts + CSV

In [ ]:
import pandas as pd
sub = pd.DataFrame(rows)
sub["transcription"] = (sub["transcription"].astype(str)
                        .str.replace(r"[\r\n]+", " ", regex=True).str.strip())
sub.loc[sub["transcription"].isin(["", "nan", "None"]), "transcription"] = "_"
sub["id"] = sub["id"].astype(str)
sub = sub.drop_duplicates(subset="id", keep="first")[["id", "language", "transcription"]]

assert list(sub.columns) == ["id", "language", "transcription"]
assert len(sub) == 41733, f"attendu 41733, obtenu {len(sub)} — vérifie les parquets/reprise"
assert sub["id"].is_unique
assert sub["language"].notna().all()
assert (sub["transcription"].str.len() > 0).all()

sub.to_csv(FINAL, index=False)
print("✅ soumission écrite ->", FINAL)
print(sub["language"].value_counts())
sub.head(8)

## 7. Traçabilité

- Config par défaut de ce notebook = **soumission finale 0.39477**.
- (0.7, 0.5) partout = soumission de contrôle **0.39923** (même moteur).
- Décomposition : glouton sans LM ≈ **0.45** (`afrivoices_soumission_greedy.ipynb`)
  → contribution du KenLM ≈ −0.055.
- 7 clips du test ont des bytes audio illisibles (tous lecteurs) → transcrits « _ »
  par conception ; ids listés dans le RAPPORT (Partie II §5).